# Case-Based Reasoning (CBR)
## Tahap 1 - Membangun Case Base
### Domain: Tindak Pidana Perdagangan Orang (TPPO)

Tahap ini bertujuan untuk:

1. Membaca dokumen PDF putusan MA RI
2. Mengekstrak teks putusan
3. Membersihkan teks
4. Menyimpan hasil ke folder data/raw
5. Membuat log proses cleaning

1.Install Library

In [1]:
!pip install pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ------------ --------------------------- 2.1/6.6 MB 11.7 MB/s eta 0:00:01
   ---------------------------- ----------- 4.7/6.6 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------  6.6/6.6 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 10.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------- ----------------------- 1.6/3.8 MB 7.0 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 8.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------------------ --------------- 2.4/3.8 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 10.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


2.Import Library

In [2]:
import os
import re
import pdfplumber
from pathlib import Path

3.Setup Folder

In [3]:
PDF_DIR = "../data/raw_pdf"
RAW_DIR = "../data/raw"
LOG_DIR = "../logs"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("Folder siap digunakan")

Folder siap digunakan


4.Cek Jumlah PDF

In [4]:
pdf_files = sorted([
    f for f in os.listdir(PDF_DIR)
    if f.endswith(".pdf")
])

print(f"Jumlah PDF ditemukan: {len(pdf_files)}")

Jumlah PDF ditemukan: 50


5.Preview Nama File

In [5]:
pdf_files[:10]

['putusan_1013_k_pid.sus_2026_20260613104634.pdf',
 'putusan_1092_k_pid_2024_20260613154610.pdf',
 'putusan_1153_pk_pid.sus_2024_20260613154739.pdf',
 'putusan_1171_pk_pid.sus_2026_20260613104008.pdf',
 'putusan_11809_k_pid.sus_2025_20260613154817.pdf',
 'putusan_11815_k_pid.sus_2025_20260613154701.pdf',
 'putusan_11_pk_pid.sus_2026_20260613154503.pdf',
 'putusan_12069_k_pid.sus_2025_20260611114339.pdf',
 'putusan_124_k_pid.sus_2026_20260613154316.pdf',
 'putusan_1297_k_pid.sus_2026_20260611061052.pdf']

6.Fungsi Ekstraksi PDF

In [6]:
def extract_pdf_text(pdf_path):
    
    full_text = ""

    try:
        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    full_text += page_text + "\n"

    except Exception as e:
        print(f"Gagal membaca {pdf_path}")
        print(e)

    return full_text

7.Fungsi Cleaning

In [7]:
def clean_text(text):

    text = text.lower()

    # hapus nomor halaman
    text = re.sub(r'halaman\s+\d+', ' ', text)

    # hapus url
    text = re.sub(r'www\.\S+', ' ', text)

    # hapus karakter aneh
    text = re.sub(r'[^a-z0-9\s.,]', ' ', text)

    # normalisasi spasi
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

8.Proses PDF → TXT + Cleaning

In [8]:
cleaning_log = []

for idx, pdf_file in enumerate(pdf_files, start=1):

    pdf_path = os.path.join(PDF_DIR, pdf_file)

    raw_text = extract_pdf_text(pdf_path)

    cleaned_text = clean_text(raw_text)

    txt_path = os.path.join(
        RAW_DIR,
        f"case_{idx:03d}.txt"
    )

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

    cleaning_log.append(
        f"{pdf_file} -> case_{idx:03d}.txt | {len(cleaned_text)} karakter"
    )

    print(f"[{idx}/{len(pdf_files)}] selesai")

[1/50] selesai
[2/50] selesai
[3/50] selesai
[4/50] selesai
[5/50] selesai
[6/50] selesai
[7/50] selesai
[8/50] selesai
[9/50] selesai
[10/50] selesai
[11/50] selesai
[12/50] selesai
[13/50] selesai
[14/50] selesai
[15/50] selesai
[16/50] selesai
[17/50] selesai
[18/50] selesai
[19/50] selesai
[20/50] selesai
[21/50] selesai
[22/50] selesai
[23/50] selesai
[24/50] selesai
[25/50] selesai
[26/50] selesai
[27/50] selesai
[28/50] selesai
[29/50] selesai
[30/50] selesai
[31/50] selesai
[32/50] selesai
[33/50] selesai
[34/50] selesai
[35/50] selesai
[36/50] selesai
[37/50] selesai
[38/50] selesai
[39/50] selesai
[40/50] selesai
[41/50] selesai
[42/50] selesai
[43/50] selesai
[44/50] selesai
[45/50] selesai
[46/50] selesai
[47/50] selesai
[48/50] selesai
[49/50] selesai
[50/50] selesai


9.Simpan Cleaning Log

In [9]:
log_path = os.path.join(
    LOG_DIR,
    "cleaning.log"
)

with open(log_path, "w", encoding="utf-8") as f:

    for row in cleaning_log:
        f.write(row + "\n")

print("Cleaning log berhasil dibuat")

Cleaning log berhasil dibuat


10.Validasi Jumlah File TXT

In [10]:
txt_files = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.endswith(".txt")
])

print("Jumlah file TXT:", len(txt_files))

Jumlah file TXT: 50


11.Validasi Keutuhan Teks

In [11]:
for file in txt_files[:10]:

    path = os.path.join(RAW_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    print(
        file,
        "| jumlah karakter:",
        len(text)
    )

case_001.txt | jumlah karakter: 23302
case_002.txt | jumlah karakter: 25284
case_003.txt | jumlah karakter: 18707
case_004.txt | jumlah karakter: 26138
case_005.txt | jumlah karakter: 30876
case_006.txt | jumlah karakter: 29255
case_007.txt | jumlah karakter: 26259
case_008.txt | jumlah karakter: 22738
case_009.txt | jumlah karakter: 27167
case_010.txt | jumlah karakter: 53132


12.Preview Hasil Cleaning

In [12]:
sample_txt = os.path.join(
    RAW_DIR,
    txt_files[0]
)

with open(sample_txt, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:3000])

a i s e n o d n i k i l b u p e a r i s g e n n o u d g n a i h k a i l m b u a p k direktori putusan mahkamah agung republik indonesia e h a putusan.mahkamahagung.go.id r a i p u t u s a n s m gnomor 1013 k pid.sus 2026 e demi keadilan berdasarkan ketuhanan yang maha esa n n o u m a h k a m a h a g u n g d g memeriksa perkara tindak pidana khusus pada tingkat kasasi yang n adimohonkan oleh penuntut umum pada kejaksaan negeri jakarta selatan, telah memutus perkara terdakwa i h nama yety herawatyk alias lis a tempat lahir jakarta i l umur tanggal lahir 47 tahun 24 februari 1978 m b jenis kelamin perempuan u a kewarganegaraan indonesia p k tempat tinggal jalan kalipasir gg. tembok nomor 24 rt h e011 rw 010, kelurahan kebon sirih, a rkecamatan menteng, jakarta pusat a i s agama islam m pekerjaan g swasta e terdakwa tersebut berada dalam tahanan rumah tahanan negara n n rutan sejak tanggal 11 november 2024 sampai dengan sekarang o u terdakwa diajukan di depan persidangan pengadilan negeri 